# **1. Coleta de dados**

Este notebook constrói a camada bruta do projeto a partir do JúpiterWeb. Cada etapa apresenta a fonte, o arquivo local e a tabela produzida.

| etapa | fonte | resultado |
|---|---|---|
| catálogo | unidade 55 | códigos de disciplinas |
| grades | cursos selecionados | semestre, natureza e carga |
| disciplinas | páginas individuais | textos, créditos e docentes |
| requisitos | cursos que oferecem cada disciplina | relações por curso |

Os HTML originais ficam em `data/raw/jupiter/`. As tabelas extraídas ficam em `data/raw/*.csv`.

In [1]:
from __future__ import annotations

import html
import re
import string
import time
import unicodedata
from datetime import datetime, timezone
from html.parser import HTMLParser
from pathlib import Path
from urllib.parse import parse_qs, urljoin, urlparse
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

### **1.1. Configuração**

A raiz é detectada automaticamente. `RAW_HTML` guarda as páginas, e `RAW_DATA` recebe as tabelas. A pausa de 0,3 segundo reduz a carga sobre o JúpiterWeb.

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DATA = ROOT / "data" / "raw"
RAW_HTML = RAW_DATA / "jupiter"
for directory in (RAW_DATA, RAW_HTML):
    directory.mkdir(parents=True, exist_ok=True)

BASE = "https://uspdigital.usp.br/jupiterweb/"
UNIT = "55"
GRADE_PAGES = [
    ("Bacharelado em Sistemas de Informação", "listarGradeCurricular?codcg=55&codcur=55051&codhab=4&tipo=N"),
    ("Licenciatura em Matemática", "listarGradeCurricular?codcg=55&codcur=55030&codhab=300&tipo=N"),
    ("Bacharelado em Matemática", "listarGradeCurricular?codcg=55&codcur=55030&codhab=200&tipo=N"),
    ("Bacharelado em Matemática Aplicada e Computação Científica", "listarGradeCurricular?codcg=55&codcur=55060&codhab=0&tipo=N"),
    ("Bacharelado em Estatística e Ciência de Dados", "listarGradeCurricular?codcg=55&codcur=55071&codhab=4&tipo=V"),
    ("Engenharia de Computação", "listarGradeCurricular?codcg=97&codcur=97001&codhab=0&tipo=N"),
    ("Licenciatura em Ciências Exatas - Habilitação em Física", "listarGradeCurricular?codcg=90&codcur=90011&codhab=204&tipo=N"),
    ("Licenciatura em Ciências Exatas - Habilitação em Matemática", "listarGradeCurricular?codcg=90&codcur=90011&codhab=404&tipo=N"),
    ("Licenciatura em Ciências Exatas - Habilitação em Química", "listarGradeCurricular?codcg=90&codcur=90011&codhab=304&tipo=N"),
    ("Licenciatura em Ciências Exatas - Núcleo Geral", "listarGradeCurricular?codcg=90&codcur=90011&codhab=104&tipo=N"),
    ("Bacharelado em Ciência de Dados", "listarGradeCurricular?codcg=55&codcur=55090&codhab=0&tipo=N"),
    ("Bacharelado em Ciências de Computação", "listarGradeCurricular?codcg=55&codcur=55041&codhab=0&tipo=N"),
]
REFRESH_RAW = False
REQUEST_INTERVAL = 0.3
USER_AGENT = "SCC5836-disciplinas-icmc (projeto academico)"

## **2. Requisições com cache**

Cada URL corresponde a um arquivo em `data/raw/jupiter/`. O cache é lido primeiro, e páginas ausentes são baixadas em ISO-8859-1 e salvas em UTF-8. `REFRESH_RAW = True` força uma nova coleta.

In [3]:
def fetch(url: str, path: Path, refresh: bool = REFRESH_RAW) -> str:
    if path.exists() and not refresh:
        return path.read_text(encoding="utf-8")
    last_error = None
    for attempt in range(4):
        try:
            with urlopen(Request(url, headers={"User-Agent": USER_AGENT}), timeout=60) as response:
                text = response.read().decode("iso-8859-1", errors="replace")
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(text, encoding="utf-8")
            time.sleep(REQUEST_INTERVAL)
            return text
        except Exception as error:
            last_error = error
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"Falha ao baixar {url}: {last_error}")


def clean(value: str) -> str:
    return " ".join(html.unescape(value or "").replace("\xa0", " ").split())


def strip_tags(value: str) -> str:
    return clean(re.sub(r"<[^>]+>", " ", value or ""))


def fold(value: str) -> str:
    value = unicodedata.normalize("NFKD", value or "")
    return " ".join(re.sub(r"[^a-z0-9]+", " ", "".join(c for c in value if not unicodedata.combining(c)).lower()).split())

## **3. Catálogo da unidade**

A lista da unidade 55 é percorrida de A a Z. Cada página `data/raw/jupiter/catalogo/letra-X.html` fornece código, nome, versão e datas, e o resultado será exportado como `data/raw/catalog.csv`.

In [4]:
def parse_catalog(page: str) -> list[dict]:
    rows = []
    for tr in re.findall(r"<TR>(.*?)</TR>", page, re.S | re.I):
        link = re.search(r"sgldis=([A-Z0-9]+)&verdis=(\d+)[^>]*>(.*?)</A>", tr, re.S | re.I)
        if not link:
            continue
        cells = [strip_tags(c) for c in re.findall(r"<TD[^>]*>(.*?)</TD>", tr, re.S | re.I)]
        rows.append({
            "code": link.group(1), "version": int(link.group(2)), "name": strip_tags(link.group(3)),
            "activation": cells[2] if len(cells) > 2 else "", "deactivation": cells[3] if len(cells) > 3 else "",
        })
    return rows


catalog_rows = []
for letter in tqdm(string.ascii_uppercase, desc="Catálogo"):
    url = urljoin(BASE, f"jupDisciplinaLista?codcg={UNIT}&tipo=D&letra={letter}")
    for row in parse_catalog(fetch(url, RAW_HTML / "catalogo" / f"letra-{letter}.html")):
        catalog_rows.append({**row, "letter": letter})

catalog = pd.DataFrame(catalog_rows).drop_duplicates("code").sort_values("code").reset_index(drop=True)
catalog["prefix"] = catalog["code"].str.extract(r"^([A-Z]+)")[0].fillna("ICMC")
display(catalog.groupby("prefix").size().rename("disciplinas").to_frame().T)
catalog.head()

Catálogo:   0%|          | 0/26 [00:00<?, ?it/s]

Catálogo: 100%|██████████| 26/26 [00:00<00:00, 1738.07it/s]

prefix,ICMC,SCC,SMA,SME,SSC
disciplinas,4,102,99,108,130


,code,version,name,activation,deactivation,letter,prefix
0,5500002,1,Seminários em Gestão Organizacional,01/01/2020,,S,ICMC
1,5500003,1,Robótica Educacional,15/07/2021,,R,ICMC
2,5500004,1,Fundamentos em Ciências de Dados,15/07/2021,,F,ICMC
3,5500005,1,Informação Profissional em Ciência de Dados,01/01/2025,,I,ICMC
4,SCC0120,3,Introdução à Ciência da Computação,01/01/2013,,I,SCC


## **4. Grades vigentes**

As grades em `data/raw/jupiter/grade-*.html` fornecem semestre ideal, natureza, créditos e carga. Disciplinas externas exigidas pelos cursos selecionados também entram no conjunto.

| tabela | conteúdo |
|---|---|
| `courses.csv` | cursos e URLs das grades |
| `curriculum.csv` | disciplinas de cada grade |
| `grade_requisites.csv` | requisitos visíveis nas grades |

In [5]:
class Rows(HTMLParser):
    def __init__(self):
        super().__init__()
        self.rows, self._cells, self._links, self._cell = [], None, None, None

    def handle_starttag(self, tag, attrs):
        if tag == "tr":
            self._cells, self._links = [], []
        elif tag in ("td", "th") and self._cells is not None:
            self._cell = []
        elif tag == "a" and self._links is not None and dict(attrs).get("href"):
            self._links.append(html.unescape(dict(attrs)["href"]))

    def handle_endtag(self, tag):
        if tag in ("td", "th") and self._cell is not None:
            self._cells.append(clean("".join(self._cell)))
            self._cell = None
        elif tag == "tr" and self._cells is not None:
            if self._cells:
                self.rows.append((self._cells, self._links))
            self._cells = self._links = None

    def handle_data(self, data):
        if self._cell is not None:
            self._cell.append(data)


def to_int(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


course_rows = []
for label, href in GRADE_PAGES:
    q = parse_qs(urlparse(href).query)
    course_rows.append({"course_id": f"{q['codcur'][0]}-{q.get('codhab', ['0'])[0]}", "course_name": label,
                        "grade_url": urljoin(BASE, href)})
courses = pd.DataFrame(course_rows)

SECTIONS = {"disciplinas obrigatorias": "obrigatória", "disciplinas optativas eletivas": "optativa eletiva",
            "disciplinas optativas livres": "optativa livre"}
RELATIONS = {"requisito": "requisito", "requisito fraco": "requisito fraco", "indicacao de conjunto": "indicação de conjunto"}
curriculum_rows, grade_requisite_rows = [], []
for course in tqdm(courses.itertuples(), total=len(courses), desc="Grades"):
    parser = Rows()
    parser.feed(fetch(course.grade_url, RAW_HTML / f"grade-{course.course_id}.html"))
    section, semester, current = "", None, ""
    for cells, links in parser.rows:
        first = cells[0] if cells else ""
        key = fold(first)
        if key in SECTIONS:
            section, semester, current = SECTIONS[key], None, ""
            continue
        period = re.match(r"(\d+)\s*o?\s*periodo ideal", key)
        if period:
            semester, current = int(period.group(1)), ""
            continue
        if any("obterDisciplina" in link for link in links) and re.fullmatch(r"[A-Z0-9]{6,8}", first.strip()):
            current = first.strip()
            numbers = [to_int(c) for c in cells[2:5]] + [None] * 3
            curriculum_rows.append({"course_id": course.course_id, "code": current, "name": cells[1] if len(cells) > 1 else current,
                                    "semester": semester, "kind": section or "outra", "credits_class": numbers[0],
                                    "credits_work": numbers[1], "workload_hours": numbers[2]})
            continue
        requisite = re.match(r"([A-Z0-9]{6,8})\s*-\s*(.+)", first)
        relation = fold(cells[1]) if len(cells) > 1 else ""
        if current and requisite and relation in RELATIONS:
            grade_requisite_rows.append({"course_id": course.course_id, "code": current,
                                         "requisite": requisite.group(1), "relation": RELATIONS[relation]})

curriculum = pd.DataFrame(curriculum_rows)
display(curriculum.groupby("course_id").agg(disciplinas=("code", "nunique")).join(courses.set_index("course_id")["course_name"]))

Grades:   0%|          | 0/12 [00:00<?, ?it/s]

Grades:  58%|█████▊    | 7/12 [00:00<00:00, 66.24it/s]

Grades: 100%|██████████| 12/12 [00:00<00:00, 70.95it/s]

,disciplinas,course_name
course_id,,
55030-200,87,Bacharelado em Matemática
55030-300,83,Licenciatura em Matemática
55041-0,140,Bacharelado em Ciências de Computação
55051-4,83,Bacharelado em Sistemas de Informação
55060-0,86,Bacharelado em Matemática Aplicada e Computaçã...
55071-4,77,Bacharelado em Estatística e Ciência de Dados
55090-0,108,Bacharelado em Ciência de Dados
90011-104,44,Licenciatura em Ciências Exatas - Núcleo Geral
90011-204,59,Licenciatura em Ciências Exatas - Habilitação ...


## **5. Páginas das disciplinas**

O conjunto de códigos combina catálogo, grades e requisitos. Cada `data/raw/jupiter/disciplinas/<código>.html` fornece nomes, créditos, carga, datas, ementa, objetivos, programa e docentes. Os textos analisados são os blocos em português.

In [6]:
SECTION_KEYS = {"ementa": "ementa", "objetivos": "objectives", "conteudo programatico": "program"}


def pre_blocks(chunk: str) -> str:
    blocks = re.findall(r"<pre>(.*?)</pre>", chunk, re.S | re.I)
    return clean(" ".join(re.sub(r"<br\s*/?>", " ", block, flags=re.I) for block in blocks))


def parse_discipline(code: str, page: str) -> dict:
    text = strip_tags(re.sub(r"<(style|script).*?</\1>", "", page, flags=re.S | re.I))
    head = re.search(r"<b>\s*Disciplina:\s*" + re.escape(code) + r"\s*-\s*(.*?)</b>.*?<span[^>]*>(.*?)</span>", page, re.S)
    spans = [strip_tags(x) for x in re.findall(r'txt_arial_10pt_black">(.*?)</span>', page, re.S)]
    number = lambda label: (lambda m: int(m.group(1)) if m else None)(re.search(label + r"\s*:\s*(\d+)", text))
    date = lambda label: (lambda m: m.group(1) if m and m.group(1) else "")(re.search(label + r":\s*(\d{2}/\d{2}/\d{4})?", text))
    marks = [(m.start(), m.end(), fold(re.sub(r"<[^>]+>", "", m.group(1)))) for m in re.finditer(r"<b>\s*(.*?)\s*</b>", page, re.S)]
    fields = dict.fromkeys(SECTION_KEYS.values(), "")
    for k, (_, end, label) in enumerate(marks):
        if label in SECTION_KEYS:
            stop = marks[k + 1][0] if k + 1 < len(marks) else len(page)
            fields[SECTION_KEYS[label]] = (fields[SECTION_KEYS[label]] + " " + pre_blocks(page[end:stop])).strip()
    start = page.find("Docente(s) Respons")
    teachers = ""
    if start >= 0:
        chunk = strip_tags(page[start:page.find("Clique", start)])
        teachers = "; ".join(clean(n) for n in re.findall(r"\d{5,8}\s*-\s*([^0-9]+?)(?=\s+\d{5,8}\s*-|$)", chunk))
    return {
        "code": code, "name": clean(head.group(1)) if head else "", "name_en": strip_tags(head.group(2)) if head else "",
        "unit": (spans + [""])[0], "department": (spans + ["", ""])[1],
        "credits_class": number("Créditos Aula"), "credits_work": number("Créditos Trabalho"),
        "workload_hours": number("Carga Horária Total"), "extension_hours": number("Carga Horária de Extensão"),
        "activation": date("Ativação"), "deactivation": date("Desativação"),
        **fields, "teachers": teachers, "url": urljoin(BASE, f"obterDisciplina?sgldis={code}"),
    }


version = dict(zip(catalog["code"], catalog["version"]))
codes = sorted(set(curriculum["code"]) | {r["requisite"] for r in grade_requisite_rows})
discipline_rows = []
for code in tqdm(codes, desc="Disciplinas"):
    suffix = f"&verdis={version[code]}" if code in version else ""
    page = fetch(urljoin(BASE, f"obterDisciplina?sgldis={code}{suffix}"), RAW_HTML / "disciplinas" / f"{code}.html")
    discipline_rows.append({**parse_discipline(code, page), "in_unit_catalog": code in version,
                            "in_current_grade": code in set(curriculum["code"])})
disciplines = pd.DataFrame(discipline_rows)
disciplines[["code", "name", "department", "workload_hours", "ementa"]].head()

Disciplinas:   0%|          | 0/582 [00:00<?, ?it/s]

Disciplinas:  21%|██        | 121/582 [00:00<00:00, 1209.07it/s]

Disciplinas:  42%|████▏     | 242/582 [00:00<00:00, 1191.19it/s]

Disciplinas:  62%|██████▏   | 362/582 [00:00<00:00, 1183.13it/s]

Disciplinas:  84%|████████▎ | 487/582 [00:00<00:00, 1208.96it/s]

Disciplinas: 100%|██████████| 582/582 [00:00<00:00, 1206.24it/s]

,code,name,department,workload_hours,ementa
0,1800040,A Ética e a Responsabilidade Social em Engenharia,Disciplinas Interdepartamentais da EESC,60.0,História da Ética. A evolução do conceito de p...
1,1800043,O Engenheiro Como Agente Ético,Disciplinas Interdepartamentais da EESC,60.0,Introdução. O agir e seus desdobramentos. O co...
2,1800118,Introdução à Engenharia Biomédica,Disciplinas Interdepartamentais da EESC,30.0,"Serão tratados, sempre dentro de uma visão int..."
3,1800119,Anatomia e Fisiologia para o Engenheiro Biomédico,Disciplinas Interdepartamentais da EESC,30.0,Anatomia geral do corpo humano. Anatomia do es...
4,1800120,Tutoria Acadêmica I,Disciplinas Interdepartamentais da EESC,45.0,(Associar o conteúdo resumido aos objetivos de...


## **6. Requisitos em todos os cursos**

Cada `data/raw/jupiter/requisitos/<código>.html` lista curso, semestre ideal e tipo da relação. A coleta segue requisitos ainda desconhecidos até que nenhuma disciplina nova apareça. O resultado será `data/raw/requisites.csv`.

In [7]:
def parse_requisites(code: str, page: str) -> list[dict]:
    rows, course = [], None
    for tr in re.findall(r"<tr[^>]*>(.*?)</tr>", page, re.S | re.I):
        text = strip_tags(tr)
        head = re.match(r"Curso:\s*(\d+)\s+(.*?)(?:\s*-\s*Período ideal:\s*(\d+))?$", text)
        if head:
            course = {"course_code": head.group(1), "course_name": head.group(2).strip(" -"), "ideal_period": to_int(head.group(3))}
            continue
        requisite = re.match(r"([A-Z0-9]{6,8})\s*-\s*(.+?)\s+(Requisito fraco|Requisito|Indicação de Conjunto)\s*$", text, re.I)
        if course and requisite:
            rows.append({"code": code, **course, "requisite": requisite.group(1), "requisite_name": requisite.group(2),
                         "relation": requisite.group(3).lower()})
    return rows


requisite_rows, done, queue, rounds = [], set(), list(codes), 0
while queue:
    rounds += 1
    for code in tqdm(queue, desc=f"Requisitos (rodada {rounds})"):
        page = fetch(urljoin(BASE, f"listarCursosRequisitos?coddis={code}"), RAW_HTML / "requisitos" / f"{code}.html")
        requisite_rows += parse_requisites(code, page)
        done.add(code)
    known = set(disciplines["code"])
    missing = sorted({r["requisite"] for r in requisite_rows} - known)
    new_rows = []
    for code in missing:
        page = fetch(urljoin(BASE, f"obterDisciplina?sgldis={code}"), RAW_HTML / "disciplinas" / f"{code}.html")
        new_rows.append({**parse_discipline(code, page), "in_unit_catalog": False, "in_current_grade": False, "via_requisite": True})
    if new_rows:
        disciplines = pd.concat([disciplines, pd.DataFrame(new_rows)], ignore_index=True)
    queue = [c for c in missing if c not in done]

disciplines["via_requisite"] = disciplines.get("via_requisite", False).fillna(False).astype(bool)
requisites = pd.DataFrame(requisite_rows)
print(f"{rounds} rodadas, {int(disciplines['via_requisite'].sum())} disciplinas incluídas por serem requisito de outra")
display(requisites.groupby("relation").size().rename("linhas").to_frame().T)
requisites.head()

Requisitos (rodada 1):   0%|          | 0/582 [00:00<?, ?it/s]

Requisitos (rodada 1):  76%|███████▌  | 442/582 [00:00<00:00, 4408.89it/s]

Requisitos (rodada 1): 100%|██████████| 582/582 [00:00<00:00, 4453.49it/s]

Requisitos (rodada 2):   0%|          | 0/35 [00:00<?, ?it/s]

Requisitos (rodada 2): 100%|██████████| 35/35 [00:00<00:00, 4533.12it/s]

Requisitos (rodada 3):   0%|          | 0/15 [00:00<?, ?it/s]

... (6 outputs left out)

Requisitos (rodada 6): 100%|██████████| 1/1 [00:00<00:00, 2941.31it/s]

6 rodadas, 54 disciplinas incluídas por serem requisito de outra


relation,indicação de conjunto,requisito,requisito fraco
linhas,73,467,752


,code,course_code,course_name,ideal_period,requisite,requisite_name,relation
0,1800043,18045,Engenharia Elétrica - Habilitação: Ênfase em E...,4,1800040,A Ética e a Responsabilidade Social em Engenharia,requisito fraco
1,1800043,18050,Engenharia Elétrica - Habilitação: Ênfase em S...,4,1800040,A Ética e a Responsabilidade Social em Engenharia,requisito fraco
2,1800043,18063,Engenharia Mecânica (integral),3,1800040,A Ética e a Responsabilidade Social em Engenharia,requisito fraco
3,1800043,18084,Engenharia de Produção (integral),4,1800040,A Ética e a Responsabilidade Social em Engenharia,requisito fraco
4,1800043,97001,Engenharia de Computação (integral),8,1800040,A Ética e a Responsabilidade Social em Engenharia,requisito fraco


## **7. Exportação e auditoria**

As tabelas extraídas são gravadas em `data/raw/`. `collection_audit.csv` registra data da coleta, cobertura dos textos e contagens usadas para conferir a etapa.

In [8]:
collected_at = datetime.fromtimestamp(max(p.stat().st_mtime for p in RAW_HTML.rglob("*.html")), timezone.utc).isoformat()
outputs = {
    "catalog.csv": catalog, "courses.csv": courses, "curriculum.csv": curriculum,
    "grade_requisites.csv": pd.DataFrame(grade_requisite_rows), "disciplines.csv": disciplines, "requisites.csv": requisites,
}
audit = pd.DataFrame([
    {"metric": "collected_at_utc", "value": collected_at},
    {"metric": "catalog_codes", "value": len(catalog)},
    {"metric": "current_grades", "value": len(courses)},
    {"metric": "disciplines_total", "value": len(disciplines)},
    {"metric": "outside_unit_catalog", "value": int((~disciplines["in_unit_catalog"]).sum())},
    {"metric": "added_as_requisite", "value": int(disciplines["via_requisite"].sum())},
    {"metric": "with_ementa", "value": int((disciplines["ementa"] != "").sum())},
    {"metric": "with_objectives", "value": int((disciplines["objectives"] != "").sum())},
    {"metric": "with_program", "value": int((disciplines["program"] != "").sum())},
    {"metric": "with_teachers", "value": int((disciplines["teachers"] != "").sum())},
    {"metric": "deactivated", "value": int((disciplines["deactivation"] != "").sum())},
    {"metric": "requisite_rows", "value": len(requisites)},
    {"metric": "requisite_courses", "value": int(requisites["course_code"].nunique())},
])
outputs["collection_audit.csv"] = audit
for name, frame in outputs.items():
    frame.to_csv(RAW_DATA / name, index=False)
display(audit)
pd.DataFrame([{"artifact": name, "rows": len(frame), "directory": str(RAW_DATA.relative_to(ROOT))} for name, frame in outputs.items()])

,metric,value
0,collected_at_utc,2026-11-15T22:33:54.246053+00:00
1,catalog_codes,443
2,current_grades,12
3,disciplines_total,636
4,outside_unit_catalog,265
5,added_as_requisite,54
6,with_ementa,621
7,with_objectives,632
8,with_program,632
9,with_teachers,495


,artifact,rows,directory
0,catalog.csv,443,data/raw
1,courses.csv,12,data/raw
2,curriculum.csv,1089,data/raw
3,grade_requisites.csv,502,data/raw
4,disciplines.csv,636,data/raw
5,requisites.csv,1292,data/raw
6,collection_audit.csv,13,data/raw
